# Silver Layer - Customers

Transform raw Bronze data into clean Silver data.

**Source:** `end-to-end_pipeline.bronze.customers`  
**Target:** `end-to-end_pipeline.silver.customers`

**Approach:** Profile → Inspect → Transform → Validate

## Step 1: Profile Bronze Data

Inspect data quality issues before transformation:

* Duplicate customer_ids
* NULL values in key fields (customer_id, customer_name, email, city)
* Whitespace in customer_id
* Inconsistent country values
* Inconsistent loyalty_status values

This single query checks all quality dimensions.

In [0]:
%sql

-- Step 1: Profile Bronze customer data

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT customer_id) AS distinct_customer_ids,
    COUNT(*) - COUNT(DISTINCT customer_id) AS duplicate_customers,

    SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END)
        AS null_customer_ids,

    SUM(CASE WHEN customer_name IS NULL THEN 1 ELSE 0 END)
        AS null_customer_names,

    SUM(CASE WHEN email IS NULL THEN 1 ELSE 0 END)
        AS null_emails,

    SUM(CASE WHEN city IS NULL THEN 1 ELSE 0 END)
        AS null_cities,

    SUM(CASE
        WHEN customer_id != TRIM(customer_id) THEN 1
        ELSE 0
    END) AS customer_id_spaces,

    COUNT(DISTINCT country)
        AS country_variations,

    COUNT(DISTINCT loyalty_status)
        AS loyalty_status_variations

FROM `end-to-end_pipeline`.bronze.customers;

total_rows,distinct_customer_ids,duplicate_customers,null_customer_ids,null_customer_names,null_emails,null_cities,customer_id_spaces,country_variations,loyalty_status_variations
2501,2500,1,0,0,1,1,0,8,4


## Step 2: Inspect Categorical Values

Review actual country and loyalty_status values to identify standardization needs:

* Country variations (DE, germany → Germany)
* Loyalty status casing (gold → Gold)

In [0]:
%sql

-- Step 2a: Inspect country values

SELECT
    country,
    COUNT(*) AS records
FROM `end-to-end_pipeline`.bronze.customers
GROUP BY country
ORDER BY records DESC;

field,value,records
country,Spain,447
country,Netherlands,434
country,France,417
country,Austria,417
country,Germany,410
country,Italy,374
country,DE,1
country,germany,1
loyalty_status,Gold,854
loyalty_status,Silver,850


In [0]:
%sql

-- Step 2b: Inspect loyalty status values

SELECT
    loyalty_status,
    COUNT(*) AS records
FROM `end-to-end_pipeline`.bronze.customers
GROUP BY loyalty_status
ORDER BY records DESC;

## Step 3: Transform to Silver

Apply all data quality fixes in one pass:

**Data Cleaning:**
* TRIM whitespace from all identifiers and text
* Standardize customer names with INITCAP
* Standardize emails to lowercase
* Standardize text fields (gender, customer_segment, city) with INITCAP

**Categorical Standardization:**
* Convert DE and germany → Germany
* Standardize loyalty_status casing (gold → Gold)

**Data Type Enforcement:**
* Convert registration_date to DATE format

**Deduplication:**
* ROW_NUMBER() to keep first occurrence per customer_id

**Quality Filters:**
* Remove NULL customer_id
* Preserve customers with missing email/city (non-critical attributes)

In [0]:
%sql

-- Step 3: Create cleaned Silver customers table

CREATE OR REPLACE TABLE `end-to-end_pipeline`.silver.customers AS

WITH cleaned AS (

    SELECT
        TRIM(customer_id) AS customer_id,

        INITCAP(TRIM(customer_name)) AS customer_name,

        LOWER(TRIM(email)) AS email,

        INITCAP(TRIM(gender)) AS gender,

        INITCAP(TRIM(customer_segment)) AS customer_segment,

        INITCAP(TRIM(city)) AS city,

        CASE
            WHEN LOWER(TRIM(country)) IN ('germany', 'de')
                THEN 'Germany'
            ELSE INITCAP(TRIM(country))
        END AS country,

        TRY_TO_DATE(
            registration_date,
            'yyyy-MM-dd'
        ) AS registration_date,

        INITCAP(TRIM(loyalty_status))
            AS loyalty_status,

        ROW_NUMBER() OVER (
            PARTITION BY TRIM(customer_id)
            ORDER BY customer_id
        ) AS row_num

    FROM `end-to-end_pipeline`.bronze.customers
)

SELECT
    customer_id,
    customer_name,
    email,
    gender,
    customer_segment,
    city,
    country,
    registration_date,
    loyalty_status

FROM cleaned

WHERE row_num = 1
  AND customer_id IS NOT NULL;

num_affected_rows,num_inserted_rows


## Step 4: Validate Silver Data

Verify all transformations were successful.

**Expected Results:**
* total_rows = 2,500
* distinct_customer_ids = 2,500
* remaining_duplicates = 0
* null_customer_ids = 0
* standardized_countries = 6
* valid registration dates
* Preserved NULL emails/cities (non-critical fields)

If any metric is unexpected, the transformation has an issue.

In [0]:
%sql

-- ============================================================
-- FINAL VALIDATION: SILVER CUSTOMERS
-- Purpose: Validate row quality, uniqueness, nulls, and allowed values
-- ============================================================

WITH validation AS (

    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT customer_id) AS distinct_customer_ids,
        COUNT(*) - COUNT(DISTINCT customer_id) AS remaining_duplicates,

        -- Null checks
        SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS null_customer_ids,
        SUM(CASE WHEN email IS NULL THEN 1 ELSE 0 END) AS null_emails,
        SUM(CASE WHEN city IS NULL THEN 1 ELSE 0 END) AS null_cities,
        SUM(CASE WHEN registration_date IS NULL THEN 1 ELSE 0 END) AS invalid_registration_dates,

        -- Distinct standardized values
        COUNT(DISTINCT country) AS standardized_countries,
        COUNT(DISTINCT loyalty_status) AS standardized_loyalty_statuses,

        -- Allowed country values
        SUM(
            CASE
                WHEN country IN (
                    'Germany',
                    'France',
                    'Spain',
                    'Italy',
                    'Austria',
                    'Netherlands'
                )
                THEN 0
                ELSE 1
            END
        ) AS invalid_countries,

        -- Allowed loyalty-status values
        SUM(
            CASE
                WHEN loyalty_status IN (
                    'Gold',
                    'Silver',
                    'Bronze'
                )
                THEN 0
                ELSE 1
            END
        ) AS invalid_loyalty_status

    FROM `end-to-end_pipeline`.silver.customers
)

SELECT
    *,

    CASE
        WHEN total_rows = 2500
            AND distinct_customer_ids = 2500
            AND remaining_duplicates = 0
            AND null_customer_ids = 0
            AND invalid_registration_dates = 0
            AND standardized_countries = 6
            AND standardized_loyalty_statuses = 3
            AND invalid_countries = 0
            AND invalid_loyalty_status = 0
        THEN 'PASS'
        ELSE 'FAIL'
    END AS validation_status

FROM validation;

total_rows,distinct_customer_ids,remaining_duplicates,null_customer_ids,null_emails,null_cities,invalid_registration_dates,standardized_countries,standardized_loyalty_statuses,invalid_countries,invalid_loyalty_status,validation_status
2500,2500,0,0,1,1,0,6,3,0,0,PASS
